# Determining the number of factors

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[0]
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import glob 
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from kneed import KneeLocator

from src.config import *
from src.file_operation import *

You can change these variables.

- ```target_dir_name```: A directory name that you saved diagnostics files (.xls) produced by EPA PMF v5
- ```data_name```: (option) A name of your data set, which will be used as a file name of a plot.

In [ ]:
target_dir_name = '100119'
data_name = 'Burnaby (BC)'

full_path = OUTPUT_IMG_DIR / 'QQ_ratio'
ensure_directory_exists(full_path)

In [ ]:
diagnostics_dir = str(RAW_DIR) + f'/diagnostics/{target_dir_name}'

# xls or xlsx
file_format = 'xls'

sheet_name='Base Runs'

all_files = glob.glob(os.path.join(diagnostics_dir, f'*.{file_format}'))

combined_df = pd.DataFrame()
dataframes = []
if file_format == 'xls':
    for file in all_files:

        # Read the entire sheet with no header
        df_full = pd.read_excel(file, sheet_name=sheet_name, engine='xlrd', header=None)
        
        # Extract factor_count from cell B5 (row 4, column 1)
        factor_count = df_full.iloc[4, 1]

        # Extract the data starting at row 9 with columns B to G (i.e., columns 1 to 6)
        df_data = df_full.iloc[9:, 1:7].copy()
        # Extract header from row 8 and use it for column names
        df_data.columns = df_full.iloc[8, 1:7].tolist()
        # !!! need reset because index of df_data is inherited from df_full !!!
        df_data.reset_index(drop=True, inplace=True)
        
        # Find the first row where 'Run #' is NaN and cut the DataFrame there
        empty_cell_index = df_data.loc[:, 'Run #'].isna().idxmax()
        df_subset_data = df_data.iloc[:empty_cell_index]
        
        # Add factor_count column
        df_subset_data.loc[:, ['factor_count']] = factor_count
        
        dataframes.append(df_subset_data)

    # Concatenate all DataFrames
    combined_df = pd.concat(dataframes, ignore_index=True)
    combined_df.sort_values(['factor_count', 'Run #'], inplace=True)

elif file_format == 'xlsx':
    # TODO: TBD
    all_files = glob.glob(diagnostics_dir + '/*.xlsx')
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        
        for fileIndex in range(len(all_files)):
        
            file_path = all_files[fileIndex]
            file_name = file_path.split('/')[-1]
            print(file_name)
        
            df = pd.read_excel(file_path, sheet_name=sheet_name, engine="openpyxl")
            factor_count = df.iloc[3, 1]
        
            df = pd.read_excel(file_path, sheet_name=sheet_name, header=8, usecols='B:G')
            empty_cell_index = df.loc[:, 'Run #'].isna().idxmax()
            subset_df = df.iloc[:empty_cell_index]
            subset_df.loc[:, ['factor_count']] = factor_count
            
            combined_df = pd.concat([dfs, subset_df])
        
# Display the first and last few rows of the combined DataFrame
display(combined_df.head(3))
display(combined_df.tail(3))


The following code will create a stats table for Q(true)/Q$_{exp}$. The table will be copied into the clipboard in the LaTeX format.

In [ ]:
# Step 1: Group and calculate stats
summary_stats = (
    combined_df.groupby('factor_count')['Q(true)/Qexp']
    .agg(['mean', 'std'])
    .reset_index()
    .sort_values('factor_count')
)

# Calculate differences between mean values of the results with n factors and (n-1) factors
for ii in np.arange(1, len(summary_stats)):
    summary_stats.loc[ii, ['diff']] = summary_stats.loc[ii - 1, 
    ['mean']].squeeze() - summary_stats.loc[ii, ['mean']].squeeze()

display(summary_stats)

# Copy the stats table to the clipboard in the LaTeX format
import pyperclip
latex_str = summary_stats.to_latex(index=False)
pyperclip.copy(latex_str.strip())


The following code will find a point of diminishing returns (i.e., knee or elbow) from the means of Q(true)/Q$_{exp}$.

In [ ]:
# Step 2: Find the knee point
knee_finder = KneeLocator(
    summary_stats['factor_count'],
    summary_stats['mean'],
    curve='convex', direction='decreasing'
)
knee = knee_finder.knee
print(f'Point of diminishing returns at = {knee}')

In [ ]:
# Step 3: Plot with error bars
plt.figure(figsize=(5, 4))
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['text.color'] = 'black'

plt.errorbar(
    summary_stats['factor_count'],
    summary_stats['mean'],
    yerr=summary_stats['std'], color='black',
    fmt='-o', capsize=5, label='Mean ± SD'
)

# Step 4: Annotate knee point
if knee is not None:
    knee_mean = summary_stats.loc[summary_stats['factor_count'] == knee, 'mean'].values[0]
    plt.axvline(x=knee, color='red', linestyle='--', label=f'Point of diminishing returns')
    plt.scatter(knee, knee_mean, color='red', zorder=5)
    # plt.text(knee, knee_mean, f'  Point of diminishing returns = {knee}', color='red', va='bottom')

# Final touches
plt.xlabel('Number of factors', fontsize=12)
plt.ylabel('Q(true)/Q$_{exp}$', fontsize=12)
plt.xticks(summary_stats['factor_count'], fontsize=12)
plt.yticks(fontsize=12)
plt.title(f'Evaluation of Model Fit Across Factor Numbers for {data_name}', fontsize=12)
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig(str(OUTPUT_IMG_DIR) + f'/QQ_ratio/{data_name}.png', dpi=300, bbox_inches='tight')
plt.show()